> **Deprecated:** These notebooks are examples only. Canonical code lives in `src/pubmed_lib/`. See `docs/` and `PROJECT.md`.


In [1]:
#| hide
%load_ext autoreload
%autoreload 2

In [30]:
#| hide
from pubmed_lib.search import *
from pubmed_lib.data import *
from nbdev.showdoc import *
from dotenv import load_dotenv, find_dotenv
from Bio import Entrez
from Bio.Entrez.Parser import DictionaryElement
import os

In [3]:
#|hide
load_dotenv(find_dotenv())

True

# pubmed_lib

> Library to search and parse data from pubmed. This library has all the functions to search for a author or affiliation, get publications, authors and some visualizations

## Install

```sh
pip install pubmed_lib
```

## How to use

This library has the basic functionalities to retreive data from pubmed search to be used with Langchain for Q&A systems, it is a simple wrapper from the Biopython

First you need to create a search object, with the default parameter you will use in the search

In [4]:
show_doc(Search, )

---

[source](https://github.com/Dmaturana81/pubmed_lib/blob/main/pubmed_lib/search.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### Search

>      Search (search_tag:str='Title/Abstract', retmax:int=200,
>              retmode:str='xml', sort:str='relevance', mindate:int|None=None,
>              maxdate:int|None=None, idlist:Optional[List[int]]=None,
>              email:str|None=None, api_key:str|None=None)

*Search class to warp the search and results*

Here we will create the object with a max number of results of 10

the search_tag correspondes to the tag where you will do the search, the options available are the following:

In [5]:
#| hide_input
for k in SEARCH_TAGS.keys():
    print(f"{k}")

Affiliation
All Fields
Article Identifier
Author
Author Identifier
EC/RN Number
First Author Name
Full Author Name
Full Investigator Name
Grant Number
Investigator
Journal
Last Author Name
Location ID
MeSH Major Topic
MeSH Subheadings
MeSH Terms
Other Term
PMID
Subset
Text Words
Title
Title/Abstract


By defaults is setup to search in Title/Abstract

In [6]:
search = Search(retmax=10)

To actually do the search, you need to call the method search and give the query

In [7]:
show_doc(Search.search)

---

[source](https://github.com/Dmaturana81/pubmed_lib/blob/main/pubmed_lib/search.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### Search.search

>      Search.search (query:str)

*It receive a query to be searched in pubmed and return the handler of the search*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| query | str | Query to be search in pubmed |

In [35]:
idlist = search.search('Bi-functional degraders in cancer')

In [36]:
idlist

['17765262', '1806659', '20230665']

to fetch the results you need to call the fetch_details method, and pass the list of pubmedIds retreive previously

In [37]:
articles = search.fetch_details(idlist)

In [56]:
author_xml = articles[0]['MedlineCitation']['Article']['AuthorList'][-1]
author_xml


DictElement({'Identifier': [], 'AffiliationInfo': [], 'LastName': 'Nordlund', 'ForeName': 'Pär', 'Initials': 'P'}, attributes={'ValidYN': 'Y'})

In [53]:
def parse_author_xml(
    autor_xml, #Xml data containing information for each author
    ):
    """
    Receive a dictionary from pubmed with the information of the Author. Retreive a Autor object with all the information parsed

    """
    # Return false if no author information found
    if 'CollectiveName' in autor_xml:
        return
    # try to parse information from XML
    try:
        #get Identifier (only orcid is used now so if they have identifier it should be the first value
        if len(autor_xml['Identifier']) > 0:
            autorID = str(autor_xml['Identifier'][0])
        else:
            autorID = ''
        #Get the affilaition details from that author, if he had
        if len(autor_xml['AffiliationInfo']) > 0:
            AFFs = ';'.join([affiliationinfo['Affiliation'] for affiliationinfo in autor_xml['AffiliationInfo']])
        else:
            AFFs = ''
        #Retrieving the name information, it is a must and should exist
        autorFN = autor_xml['ForeName']
        autorLN = autor_xml['LastName']
        autorIN = autor_xml['Initials']
        name = autorFN + ' ' + autorLN
        #Need to parse affiliation to get more details
        print("entering")
        affiliation_parsed = parse_affiliation(AFFs).model_dump() if AFFs != "" else {}
        print(affiliation_parsed)
        # emails = parse_email(AFFs) 
        data = {
            'Fname': autorFN,
            'Lname': autorLN,
            # 'emails': emails,
            'affiliations': AFFs, 
            'identifier': autorID,
            'name': name, 
            'initials': autorIN,
            'affiliation_parsed': affiliation_parsed
               }  #
        data.update(affiliation_parsed)
        return data
        # return Autor.model_validate(data)

    except ValueError as e:
        print('not possible to get info value error')
        print(e)
        return
    except OSError as err:
        print("OS Error: {0}".format(err))
        return
    except:
        print('error en parsing nor validated')
        return

In [54]:
parse_author_xml(author_xml)

entering
error en parsing nor validated


In [50]:
author_xml.attributes

{'ValidYN': 'Y'}

In [27]:
import json

This will give you the xml data retreived from pubmed

In order to retreive the parsed resutls, you should use the method results

In [11]:
show_doc(Search.results)

---

[source](https://github.com/Dmaturana81/pubmed_lib/blob/main/pubmed_lib/search.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### Search.results

>      Search.results (query:str)

*Method that do the search and retrieve a generator with all the infomration of the articles*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| query | str | Term to be queried in pubmed |
| **Returns** | **list** |  |

In [15]:
results = search.results('Bi-functional degraders in cancer')

entering
error en parsing nor validated
entering
{}
entering
{}
entering
{}
entering
error en parsing nor validated
entering
{}
entering
{}
entering
{}
entering
error en parsing nor validated
entering
{}
entering
{}
entering
{}
entering
{}
entering
{}
entering
{}


In [16]:
res = list(results)

In [17]:
res[0]

Result(pubmed='17765262', pmc=None, doi='10.1016/j.jmb.2006.12.009', pii='S0022-2836(06)01668-8', abstract='We have determined the crystal structure of the bi-functional deaminase/reductase enzyme from Escherichia coli (EcRibD) that catalyzes two consecutive reactions during riboflavin biosynthesis. The polypeptide chain of EcRibD is folded into two domains where the 3D structure of the N-terminal domain (1-145) is similar to cytosine deaminase and the C-terminal domain (146-367) is similar to dihydrofolate reductase. We showed that EcRibD is dimeric and compared our structure to tetrameric RibG, an ortholog from Bacillus subtilis (BsRibG). We have also determined the structure of EcRibD in two binary complexes with the oxidized cofactor (NADP(+)) and with the substrate analogue ribose-5-phosphate (RP5) and superposed these two in order to mimic the ternary complex. Based on this superposition we propose that the invariant Asp200 initiates the reductive reaction by abstracting a proton

In [20]:
res[2].dict()

{'pubmed': '20230665',
 'pmc': None,
 'doi': None,
 'pii': None,
 'abstract': 'To prepare and characterize streptavidin-tagged murine interleukin-15 fusion proteins.. pET24a-SA-L-mIL15 and pET21a-mIL15-L-SA plasmids were constructed and expressed in Rosetta (DE3) host bacteria to generate SA/mIL15 fusion proteins. SA-mIL15 fusion protein was purified through the Ni-NTA affinity chromatography, and mIL15-SA fusion protein through anion exchange chromatography, followed by refolding. The efficiency of surface modification of the fusion proteins on the biotinylated RM-1 tumor cells was evaluated by a flow cytometer. MTT method was used to evaluate the proliferating effect of SA/mIL15 fusion proteins on mouse spleen lymphocytes stimulated by ConA.. Both SA-mIL15 and mIL15-SA fusion proteins were highly expressed in Rosetta (DE3) at about 20% of total bacterial proteins. They exhibited the bi-functionality: proliferation-promoting activity of mIL15 on mouse spleen lymphocytes with the speci